In [ ]:
# Importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
#from skopt import BayesSearchCV
from sklearn.model_selection import cross_validate
from sklearn.metrics import (make_scorer, average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix)
import shap

In [ ]:
# Reading in dataset

df = pd.read_csv('complete_df.csv')

In [ ]:
# Creating x and y variables
X = df.drop(columns= 'ad')
y = df['ad']

In [ ]:
# Split into test and training set

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# Assessing how many participants in training and test set

train_patients = len(X_train)
test_patients = len(X_test)

print(f'training set = {train_patients}')
print(f'test set = {test_patients}')

## **Missing Values**

In [ ]:
na_training = X_train.isna().sum().sum()
na_test = X_test.isna().sum().sum()

print(f'training NaNs = {na_training}')
print(f'test NaNs = {na_test}')

In [ ]:
na_training = y_train.isna().sum().sum()
na_test = y_test.isna().sum().sum()

print(f'training NaNs = {na_training}')
print(f'test NaNs = {na_test}')

## **Testing for other versions of NA**

### e.g. measurements of -1

In [ ]:
na_training = (X_train <= 0).sum().sum()
na_test = (X_test <= 0).sum().sum()

print(f'training NaNs = {na_training}')
print(f'test NaNs = {na_test}')

# No abnormal values

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

# nothing abnormal

In [ ]:
X_train.columns

## **Quick visualisation**

In [ ]:
fig, axes = plt.subplots(9, 3, figsize=(15, 25))
axes = axes.flatten()

for i, col in enumerate(X_train.columns):
    axes[i].hist(X_train[col].dropna(), bins=30, edgecolor='black')
    axes[i].set_title(col, fontsize=9)

plt.tight_layout()
plt.show()

## **Hyperparameter selection + Scaling**

In [ ]:
# Defining the cv parameters
cv = StratifiedKFold(n_splits=5)

In [ ]:
# Creating the parameter space for each model to test

searches = {}

# Logistic regression
searches['logreg'] = (
    Pipeline([('scaler', StandardScaler()),
              ('model', LogisticRegression(l1_ratio=1, solver='saga',
                                           max_iter=5000, class_weight='balanced'))]),
    {'model__l1_ratio': [1, 0, 0.5],
     'model__C': (1e-3, 1e2, 'log-uniform')}
)

# Random Forest
searches['rf'] = (
    Pipeline([('scaler', StandardScaler()),
              ('model', RandomForestClassifier(class_weight='balanced', random_state=42))]),
    {'model__n_estimators': (200, 800),
     'model__max_depth': (2, 20),
     'model__min_samples_leaf': (1, 20),
     'model__max_features': (0.1, 1.0)}
)

# Gradient Boosting
searches['gb'] = (
    Pipeline([('scaler', StandardScaler()),
              ('model', GradientBoostingClassifier(random_state=42))]),
    {'model__n_estimators': (200, 800),
     'model__max_depth': (2, 8),
     'model__learning_rate': (1e-3, 3e-1, 'log-uniform'),
     'model__subsample': (0.5, 1.0),
     'model__max_features': (0.5, 1.0)}
)

# SVM
searches['svm'] = (
    Pipeline([('scaler', StandardScaler()),
              ('model', SVC(class_weight='balanced', random_state=42))]),
    {'model__C': (1e-2, 1e2, 'log-uniform'),
     'model__gamma': (1e-4, 1e0, 'log-uniform'),
     'model__kernel': ['rbf', 'linear', 'poly', 'sigmoid']}
)

In [ ]:
# Hyperparameter tuning- retrieving the best hyperparameters tunings for the best model

fitted = {}
for name, (pipeline, space) in searches.items():
    model = BayesSearchCV(pipeline, space, n_iter=40, cv=cv,
                        scoring='average_precision', n_jobs=-1, random_state=42) # creates bayessearch instance
    model.fit(X_train, y_train) # executes on model
    fitted[name] = model # adds to dict
    print(f"{name}: Best score: {model.best_score_:.3f} | best parameters: {model.best_params_}")

In [ ]:
# Assessing the full metrics to decide on the best model with parameters

# Creating a dictionary of the metrics I want
scoring = {
    'avg_precision': 'average_precision',
    'roc_auc': 'roc_auc',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'balanced_accuracy': 'balanced_accuracy',
    'specificity': make_scorer(recall_score, pos_label=0),
}

# Empty list of rows to fill in for the dataframe
rows = []
for name, model in fitted.items(): # iterates through each model
    est = model.best_estimator_          # best pipeline from the search
    cvres = cross_validate(est, X_train, y_train, cv=cv,
                           scoring=scoring, n_jobs=-1) # cross validating to get SDs- probs could have done this earlier but last code took a while to run so don't really want to rerun it
    row = {'model': name} # creates row for model
    for metric in scoring:
        row[metric] = cvres[f'test_{metric}'].mean() # adds mean for each scoring metric
        row[f'{metric}_std'] = cvres[f'test_{metric}'].std() # adds std for each scoring metric
    rows.append(row) # adds it to big list

comparison = pd.DataFrame(rows).set_index('model') # turns into dataframe


In [ ]:
comparison

# random forest has best balance of precision and recall

## **Applying to test set**

In [ ]:
# Function to calculate CIs for the probability metrics

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def bootstrap_all(model, X_test, y_test, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_test)
    probs  = model.predict_proba(X_test)[:, 1]
    preds  = model.predict(X_test)

    # metrics that use probabilities vs labels
    prob_metrics  = {'AP': average_precision_score, 'AUC': roc_auc_score}
    label_metrics = {'Precision': precision_score, 'Recall': recall_score,
                     'F1': f1_score, 'Specificity': specificity_score}

    # point estimates
    results = {}
    for name, fn in prob_metrics.items():
        results[name] = [fn(y_true, probs)]
    for name, fn in label_metrics.items():
        results[name] = [fn(y_true, preds)]

    # bootstrap distributions
    boot = {name: [] for name in list(prob_metrics) + list(label_metrics)}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        for name, fn in prob_metrics.items():
            boot[name].append(fn(y_true[idx], probs[idx]))
        for name, fn in label_metrics.items():
            boot[name].append(fn(y_true[idx], preds[idx]))

    # assemble point + CI
    out = {}
    for name in boot:
        pt = results[name][0]
        lo, hi = np.percentile(boot[name], [2.5, 97.5])
        out[name] = (round(pt, 3), round(lo, 3), round(hi, 3))
        print(f"{name:12s}: {pt:.3f} (95% CI {lo:.3f}–{hi:.3f})")
    return out


In [ ]:
# Recreating the best models as kernel was reset- used the hyperparameters from the input

models = {
    'Logistic regression': LogisticRegression(
        C=0.035649432898686946,
        l1_ratio=1,
        penalty='elasticnet',
        solver='saga',
        max_iter=5000,
        class_weight='balanced',
        random_state=42,
    ),
    'Random forest': RandomForestClassifier(
        max_depth=20,
        max_features=0.9607395461141658,
        min_samples_leaf=6,
        n_estimators=203,
        class_weight='balanced',
        random_state=42,
    ),
    'Gradient boosting': GradientBoostingClassifier(
        learning_rate=0.0026786249792017263,
        max_depth=8,
        max_features=0.5,
        n_estimators=800,
        subsample=0.5,
        random_state=42,
    ),
    'SVM': SVC(
        C=1.4473325389483476,
        gamma=0.008947938504981955,
        kernel='rbf',
        probability=True,
        class_weight='balanced',
        random_state=42,
    ),
}

fitted = {}
for name, clf in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    print(f"--- {name} ---")
    results = bootstrap_all(pipe, X_test, y_test)

In [ ]:
np.save('y_test.npy', np.asarray(y_test))
np.save('score_all.npy',    fitted['Random forest'].predict_proba(X_test)[:, 1])

## **SHAP analysis**

In [ ]:
best_rf = fitted['Random forest']

In [ ]:
# SHAP analysis to assess the biggest contributing proteins

# pull the two pieces of the pipeline apart
scaler = best_rf.named_steps['scaler']
rf     = best_rf.named_steps['model']

# Manually scaling
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Changing back to dataframe for easier use
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns,
                             index=X_test.index)

In [ ]:
# Changing names of demographic variables for figure readability

X_test_scaled = X_test_scaled.rename(columns={'age_at_visit': 'Age'})
X_test_scaled = X_test_scaled.rename(columns={'sex': 'Sex'})
X_test_scaled = X_test_scaled.rename(columns={'age_squared': 'Age²'})
X_test_scaled = X_test_scaled.rename(columns={'age_x_sex': 'Age-by-sex interaction'})

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_scaled)

In [ ]:
shap_values = shap_values[:, :, 1]

In [ ]:
np.save('shap_all.npy', shap_values)
np.save('shap_features.npy', np.array(X_test_scaled.columns))

In [ ]:
# Writing a function that shortens the long name

def short_name(label):
    if label.endswith(')') and '(' in label:
        return label[label.rfind('(') + 1 : -1]
    return label

In [ ]:
# Applying function
X_test_short = X_test_scaled.rename(columns=short_name)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_short,
    cmap= 'viridis',
    max_display= 25 # getting all proteins shown
)

In [ ]:
np.save('y_test.npy', y_test.values)

In [ ]:
np.save('score_all.npy',    y_score)